# LSTM Next-Word Language Model

An end-to-end PyTorch pipeline that trains a word-level LSTM to predict the next
token in a sequence. The notebook covers text preprocessing, vocabulary
construction, sequence generation, batching, model definition, GPU training,
and inference.

**Pipeline overview**

1. Imports
2. Sample corpus
3. Download NLTK resources
4. Tokenize the document
5. Build the vocabulary
6. Split the corpus into sentences
7. Encode sentences as index sequences
8. Generate training sequences
9. Pad sequences to equal length
10. Build feature / target tensors
11. Dataset and DataLoader
12. Define the LSTM model
13. Instantiate the model and select the device
14. Train on the GPU
15. Generate next-word predictions
16. Generate text autoregressively
17. Evaluate accuracy

## 1. Imports

Load the core libraries: **PyTorch** (model, tensors, training), **NumPy**, **NLTK** (word tokenization), and `Counter` for frequency counts.

In [22]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from collections import Counter  # Counts frequency of elements in a list or text.
from torch.utils.data import Dataset, DataLoader
from nltk.tokenize import word_tokenize  # Splits text into individual words/tokens.
import nltk  # NLP tasks: tokenization, stemming, stopword removal, etc.

## 2. Sample Corpus

The raw training text — a short story. The model learns next-word patterns from this single document.

In [23]:
document = """The Last Train Journey

On a cold winter evening, Daniel arrived at the old railway station carrying a small leather bag.
The station was nearly empty except for a few passengers waiting quietly on wooden benches.
A loud whistle echoed through the air as the final train of the night slowly entered the platform.

Daniel checked the ticket in his pocket and walked toward the train.
The metal doors opened with a creaking sound and warm air rushed outside.
Inside the train, yellow lights flickered softly above rows of empty seats.

He chose a seat beside the window and placed his bag carefully near his feet.
As the train began to move, the city lights slowly disappeared behind thick clouds of fog.
Snow started falling gently outside while distant mountains appeared under the moonlight.

Across the aisle sat an elderly woman reading an old book with a blue cover.
Every few minutes she looked outside the window as if searching for something familiar.
Daniel noticed a silver necklace around her neck shaped like a small compass.

After several hours, the train stopped at a remote station surrounded by tall pine trees.
Very few people entered or left the train at that place.
A strange silence covered the station as the doors closed once again.

Curious about the woman, Daniel finally started a conversation.
The woman introduced herself as Eleanor and explained that she was returning to her hometown after many years.
She said the town was hidden deep within the northern mountains and was known for its beautiful frozen lake.

As the journey continued, Eleanor shared stories from her childhood.
She described colorful festivals, music performances, and lanterns floating across the lake during winter nights.
Daniel listened carefully while the train moved through snowy valleys and dark forests.

Near midnight, the train suddenly slowed down because heavy snow had covered the tracks ahead.
Passengers became nervous and began discussing possible delays.
Conductors moved through the cabins trying to calm everyone.

Daniel looked outside and saw workers removing snow under bright floodlights.
The freezing wind shook the train slightly while snow continued falling from the dark sky.
Despite the delay, Eleanor remained calm and continued telling her stories.

Several hours later, the tracks were finally cleared and the train resumed its journey.
The passengers sighed with relief as warm coffee was served throughout the cabins.
Daniel and Eleanor continued talking about travel, history, and forgotten places around the world.

Just before sunrise, the train reached the mountain town.
The station was small but beautifully decorated with glowing lamps and snow-covered rooftops.
People wearing thick winter coats walked slowly across the icy streets.

Before leaving, Eleanor handed Daniel a folded piece of paper.
Inside it was a drawing of the frozen lake along with a short message thanking him for the conversation.
Daniel smiled and carefully placed the paper inside his notebook.

As the train departed once again, Daniel stood on the platform watching the sunrise over the mountains.
The cold air, quiet streets, and distant church bells created a peaceful atmosphere he would never forget.
Years later, Daniel would still remember that mysterious winter journey and the stories shared by Eleanor on the last train of the night.
"""

## 3. Download NLTK Resources

NLTK ships without data files. Download the `punkt` tokenizer models; newer NLTK releases also require the `punkt_tab` tables.

In [24]:
# punkt: core word/sentence tokenizer models
nltk.download('punkt')
# punkt_tab: tokenizer tables required by newer NLTK versions
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /Users/hossain/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/hossain/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## 4. Tokenize the Document

Lowercase the text for consistency and split it into word-level tokens. Lowercasing collapses casing variants (e.g. *The* / *the*) into one token.

In [25]:
# Lowercase first so casing variants map to the same token, then split into words
tokens = word_tokenize(document.lower())
print(tokens[:20])  # Peek at the first 20 tokens

['the', 'last', 'train', 'journey', 'on', 'a', 'cold', 'winter', 'evening', ',', 'daniel', 'arrived', 'at', 'the', 'old', 'railway', 'station', 'carrying', 'a', 'small']


## 5. Build the Vocabulary

Assign every unique token an integer ID. Index `0` is reserved for `<unk>`, the placeholder for unknown / out-of-vocabulary words encountered later.

In [26]:
vocab = {'<unk>': 0}  # Index 0 reserved for unknown / out-of-vocabulary words
# Counter(tokens).keys() yields each unique token once
for token in Counter(tokens).keys():
    if token not in vocab:
        vocab[token] = len(vocab)  # Next free index = current dict size

print(list(vocab.items())[:5])  # First 5 (word -> index) pairs
len(vocab)  # Total vocabulary size

[('<unk>', 0), ('the', 1), ('last', 2), ('train', 3), ('journey', 4)]


284

## 6. Split the Document into Sentences

Break the corpus on newlines to recover individual sentences. Blank lines are filtered out so they do not produce empty sequences downstream.

In [27]:
# Split on newlines; drop blank lines so they don't become empty token lists
input_sentences = [s for s in document.split('\n') if s.strip()]
input_sentences[:5]

['The Last Train Journey',
 'On a cold winter evening, Daniel arrived at the old railway station carrying a small leather bag.',
 'The station was nearly empty except for a few passengers waiting quietly on wooden benches.',
 'A loud whistle echoed through the air as the final train of the night slowly entered the platform.',
 'Daniel checked the ticket in his pocket and walked toward the train.']

## 7. Encode Sentences as Index Sequences

Convert each sentence into a list of vocabulary indices using `text_to_indices`. Unknown tokens fall back to the `<unk>` index.

In [28]:
def text_to_indices(sentence, vocab):
    """Map a list of word tokens to their vocab indices (<unk> if unseen)."""
    numerical_sentence = []
    for token in sentence:
        if token in vocab:
            numerical_sentence.append(vocab[token])      # Known word -> its index
        else:
            numerical_sentence.append(vocab['<unk>'])     # Unknown word -> <unk> index
    return numerical_sentence

# Tokenize and encode every sentence into a list of indices
input_numerical_sentences = []
for sentence in input_sentences:
    tokens = word_tokenize(sentence.lower())
    input_numerical_sentences.append(text_to_indices(tokens, vocab))

input_numerical_sentences[:5]

[[1, 2, 3, 4],
 [5, 6, 7, 8, 9, 10, 11, 12, 13, 1, 14, 15, 16, 17, 6, 18, 19, 20, 21],
 [1, 16, 22, 23, 24, 25, 26, 6, 27, 28, 29, 30, 5, 31, 32, 21],
 [6, 33, 34, 35, 36, 1, 37, 38, 1, 39, 3, 40, 1, 41, 42, 43, 1, 44, 21],
 [11, 45, 1, 46, 47, 48, 49, 50, 51, 52, 1, 3, 21]]

## 8. Generate Training Sequences

For each encoded sentence, emit every prefix of length ≥ 2. Each prefix is one training example: the model sees all tokens but the last and learns to predict the final token.

In [29]:
# Turn each encoded sentence into growing prefixes (next-word-prediction samples)
training_sequence = []
for sentence in input_numerical_sentences:
    for i in range(1, len(sentence)):
        training_sequence.append(sentence[:i + 1])  # Prefix of length i + 1

training_sequence[:5]

[[1, 2], [1, 2, 3], [1, 2, 3, 4], [5, 6], [5, 6, 7]]

## 9. Pad Sequences to Equal Length

Neural networks need rectangular batches. Left-pad every sequence with `0` up to the longest length so all rows share the same shape.

In [30]:
max_len = max(len(seq) for seq in training_sequence)  # Length of the longest sequence

# Left-pad each sequence with 0s so every row has the same length
padded_training_sequence = []
for sequence in training_sequence:
    padded_training_sequence.append([0] * (max_len - len(sequence)) + sequence)

padded_training_sequence[:5]

[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 3],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 3, 4],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 6],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 6, 7]]

## 10. Build Feature and Target Tensors

Convert the padded sequences into a `long` tensor, then split each row: all tokens except the last form the input **X**, and the last token is the target **y**.

In [31]:
# Convert the list of lists into one integer tensor (token indices must be long)
padded_training_sequence = torch.tensor(padded_training_sequence, dtype=torch.long)

X = padded_training_sequence[:, :-1]  # Inputs: every token except the last
y = padded_training_sequence[:, -1]   # Target: the final token to predict

X.shape, y.shape

(torch.Size([555, 24]), torch.Size([555]))

## 11. Dataset and DataLoader

Wrap **X** / **y** in a custom `Dataset` and feed it through a `DataLoader` for shuffled mini-batching. The sanity check prints the first batch's shapes.

In [32]:
class CustomDataset(Dataset):
    def __init__(self, X, y):
        self.X = X  # Input sequences
        self.y = y  # Target tokens

    def __len__(self):
        return self.X.shape[0]  # Number of samples

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]  # One (input, target) pair


dataset = CustomDataset(X, y)
# Batch the data and reshuffle every epoch for better training
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# Sanity check: inspect the shapes of the first batch
for inputs, targets in dataloader:
    print('Input batch shape:', inputs.shape)
    print('Output batch shape:', targets.shape)
    break

Input batch shape: torch.Size([32, 24])
Output batch shape: torch.Size([32])


## 12. Define the LSTM Model

The network has three layers:

- **Embedding** — maps each token index to a 100-dimensional dense vector.
- **LSTM** — processes the sequence and produces a 150-dimensional hidden state.
- **Linear** — projects the final hidden state to vocabulary-sized logits.

The LSTM returns three things: the per-timestep hidden states, the **final hidden state**, and the **final cell state**. Only the final hidden state is used for the next-token prediction.

In [33]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 100)   # Token index -> 100-dim vector
        self.lstm = nn.LSTM(100, 150, batch_first=True)  # 100-dim input -> 150-dim hidden
        self.fc = nn.Linear(150, vocab_size)             # Hidden state -> vocab logits

    def forward(self, x):
        embedded = self.embedding(x)  # (batch, seq_len, 100)
        # LSTM returns all hidden states plus the final (hidden, cell) states
        intermediate_hidden_state, (final_hidden_state, final_cell_state) = self.lstm(embedded)
        # Use the final hidden state to predict the next token
        output = self.fc(final_hidden_state.squeeze(0))  # (batch, vocab_size)
        return output

## 13. Instantiate the Model and Select the Device

Build the model sized to the vocabulary, then choose the best compute device. On Apple Silicon the GPU is exposed through the **MPS** backend (CUDA is NVIDIA-only). Moving the model to `device` places its parameters on the GPU.

In [34]:
model = LSTMModel(len(vocab))  # Build the model sized to the vocabulary

In [35]:
# Pick the best available device.
# Mac (Apple Silicon) GPU = MPS backend; CUDA is NVIDIA-only and never available on macOS.
device = torch.device(
    'mps' if torch.backends.mps.is_available()
    else 'cuda' if torch.cuda.is_available()
    else 'cpu'
)
print('Using device:', device)

model = model.to(device)  # Move model parameters onto the GPU
next(model.parameters()).device  # Confirm where the model now lives

Using device: mps


device(type='mps', index=0)

## 14. Train on the GPU

Define the loss (`CrossEntropyLoss`) and optimizer (`Adam`), then run the training loop. Each batch is moved to `device` with `.to(device)` so all computation happens on the GPU alongside the model.

In [36]:
epochs = 50
criterion = nn.CrossEntropyLoss()                     # Loss for multi-class token prediction
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam optimizer

for epoch in range(epochs):
    total_loss = 0.0
    for inputs, targets in dataloader:
        # Move this batch onto the same device as the model (GPU)
        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()            # Clear gradients from the previous step
        outputs = model(inputs)          # Forward pass (runs on GPU)
        loss = criterion(outputs, targets)
        loss.backward()                  # Backpropagation
        optimizer.step()                 # Update weights

        total_loss += loss.item()

    # Average loss across all batches this epoch
    print(f'Epoch {epoch + 1}/{epochs} - loss: {total_loss / len(dataloader):.4f}')

Epoch 1/50 - loss: 5.6056
Epoch 2/50 - loss: 5.2968
Epoch 3/50 - loss: 4.8511
Epoch 4/50 - loss: 4.5441
Epoch 5/50 - loss: 4.2916
Epoch 6/50 - loss: 3.9801
Epoch 7/50 - loss: 3.7399
Epoch 8/50 - loss: 3.3972
Epoch 9/50 - loss: 3.1277
Epoch 10/50 - loss: 2.8625
Epoch 11/50 - loss: 2.5988
Epoch 12/50 - loss: 2.3490
Epoch 13/50 - loss: 2.0937
Epoch 14/50 - loss: 1.8830
Epoch 15/50 - loss: 1.6874
Epoch 16/50 - loss: 1.4984
Epoch 17/50 - loss: 1.3254
Epoch 18/50 - loss: 1.1816
Epoch 19/50 - loss: 1.0464
Epoch 20/50 - loss: 0.9425
Epoch 21/50 - loss: 0.8466
Epoch 22/50 - loss: 0.7531
Epoch 23/50 - loss: 0.6713
Epoch 24/50 - loss: 0.6085
Epoch 25/50 - loss: 0.5677
Epoch 26/50 - loss: 0.5104
Epoch 27/50 - loss: 0.4581
Epoch 28/50 - loss: 0.4186
Epoch 29/50 - loss: 0.3954
Epoch 30/50 - loss: 0.3723
Epoch 31/50 - loss: 0.3433
Epoch 32/50 - loss: 0.3208
Epoch 33/50 - loss: 0.2926
Epoch 34/50 - loss: 0.2844
Epoch 35/50 - loss: 0.2616
Epoch 36/50 - loss: 0.2468
Epoch 37/50 - loss: 0.2464
Epoch 38/5

## 15. Generate Next-Word Predictions

Given a text prompt, tokenize and encode it, left-pad to `max_len`, run a forward pass, and append the highest-scoring vocabulary token. The same `text_to_indices` and padding logic used in training is reused here so inputs match the training distribution.

In [37]:
def prediction(model, vocab, text):
    """Predict the single next word for a text prompt and append it."""
    tokenized_text = word_tokenize(text.lower())                 # Tokenize the prompt
    text_indices = text_to_indices(tokenized_text, vocab)        # Encode to vocab indices
    padded_text = [0] * (max_len - len(text_indices)) + text_indices  # Left-pad to max_len
    # Shape (1, seq_len); move to the model's device
    input_tensor = torch.tensor(padded_text, dtype=torch.long).unsqueeze(0).to(device)
    output = model(input_tensor)                                 # Logits, shape (1, vocab_size)
    _, index = torch.max(output[0], dim=0)                       # Index of the highest logit
    return text + ' ' + list(vocab.keys())[index]               # Append the predicted word

In [38]:
prediction(model, vocab, "the city lights slowly disappeared ")

'the city lights slowly disappeared  behind'

## 16. Generate Text Autoregressively

Feed the model its own output: starting from a seed phrase, predict one word, append it, and repeat. Each iteration extends the sequence by a single token, producing a generated continuation.

In [42]:
num_tokens = 10
input_text = "the train stopped at a remote station"

# Repeatedly predict and append the next word
for _ in range(num_tokens):
    input_text = prediction(model, vocab, input_text)
    print(input_text)

the train stopped at a remote station surrounded
the train stopped at a remote station surrounded by
the train stopped at a remote station surrounded by tall
the train stopped at a remote station surrounded by tall pine
the train stopped at a remote station surrounded by tall pine trees
the train stopped at a remote station surrounded by tall pine trees .
the train stopped at a remote station surrounded by tall pine trees . .
the train stopped at a remote station surrounded by tall pine trees . . .
the train stopped at a remote station surrounded by tall pine trees . . . .
the train stopped at a remote station surrounded by tall pine trees . . . . .


## 17. Evaluate Accuracy

Measure next-token accuracy over the full dataset: switch the model to eval mode, disable gradients, and count how often the top prediction matches the target token. Note this is training-set accuracy (no held-out split), so a high value mostly reflects memorization of the small corpus.

In [40]:
def calculate_accuracy(model, dataloader, device):
    """Fraction of samples whose top-1 predicted token matches the target."""
    model.eval()                       # Evaluation mode (disables dropout etc.)
    correct = 0
    total = 0
    with torch.no_grad():              # No gradient tracking during evaluation
        for inputs, targets in dataloader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            outputs = model(inputs)                    # Logits
            _, predicted = torch.max(outputs, dim=1)   # Top-1 predicted token
            correct += (predicted == targets).sum().item()
            total += targets.size(0)
    return correct / total if total > 0 else 0

In [41]:
calculate_accuracy(model, dataloader, device)

0.9693693693693693